# CP5403 A2 – Socratic Tutor
## 03 · Tutor app: RAG + LoRA + multi-turn conversation

**What this notebook does:**
1. Load the FAISS index from `01` and the LoRA adapter from `02`
2. Build prompts in **exactly the same format as the training data** (`[Course context]` / `[Conversation so far]` / `[Student]`)
3. Run a multi-turn tutor session (history is managed in code, not by the model)
4. Scripted demo conversation for the Week 9 presentation
5. Interactive chat, with retrieved sources and timing shown for every turn

The `tutor_reply(..., mode=...)` function supports four modes, reused in `04_evaluation`:

| mode | RAG | LoRA adapter |
|---|---|---|
| `base` | ✗ | ✗ |
| `rag` | ✓ | ✗ |
| `ft` | ✗ | ✓ |
| `hybrid` | ✓ | ✓ ← the final system |

**Runtime:** T4 GPU.
**Needs:** `index/cp5403.faiss`, `index/chunks.json`, `models/socratic_lora/`

## 1. Install packages, check GPU, mount Drive

In [5]:
!pip install -q -U peft bitsandbytes accelerate sentence-transformers faiss-cpu

import torch
assert torch.cuda.is_available(), "No GPU – switch the runtime to T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

GPU: Tesla T4
Mounted at /content/drive


## 2. Configuration
`SYSTEM_PROMPT` must be **identical** to the one in the training data, otherwise the model sees a different input format at inference time.

Context and generation limits are kept small on purpose: the model was fine-tuned on short, clean context snippets, so long raw slide text at inference time causes off-topic and unsupported replies.

In [6]:
import os, json, time, re

BASE        = "/content/drive/MyDrive/CP5403_A2"
INDEX_DIR   = f"{BASE}/index"
ADAPTER_VERSION = "v2"                                   # switch to "" to reload the v1 adapter for comparison
ADAPTER_DIR = f"{BASE}/models/socratic_lora_{ADAPTER_VERSION}" if ADAPTER_VERSION else f"{BASE}/models/socratic_lora"
LOG_DIR     = f"{BASE}/transcripts/{ADAPTER_VERSION or 'v1'}"
os.makedirs(LOG_DIR, exist_ok=True)

MODEL_ID    = "microsoft/Phi-3-mini-4k-instruct"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K             = 2     # chunks retrieved per turn – must match 02 (v2)
THRESHOLD         = 0.35  # from the 01 retrieval test
CHUNK_WORDS       = 60    # must match 02 (v2)
MAX_HISTORY_TURNS = 3
MAX_NEW_TOKENS    = 130

SYSTEM_PROMPT = ("You are a Socratic tutor for the JCU course CP5403 Large Language Models. "
"Use only the course context provided. Do not give the full answer straight away: ask one or two guiding questions, "
"give small hints, and build on the student's previous reply. If the student is correct, confirm it, summarise briefly and extend with one question. "
"If the student is stuck, simplify with an example. If the course context does not cover the question, say so instead of guessing. "
"Answer simple course-logistics questions directly. Never write assessment work for the student. Keep replies under 120 words.")

# sanity check against the training file
with open(f"{BASE}/data/socratic_train.jsonl", encoding="utf-8") as f:
    train_system = json.loads(f.readline())["messages"][0]["content"]
assert train_system == SYSTEM_PROMPT, "System prompt differs from the training data!"
assert os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"), f"Adapter not found: {ADAPTER_DIR}"
print("✅ System prompt matches training data")
print("Adapter:", ADAPTER_DIR)

✅ System prompt matches training data
Adapter: /content/drive/MyDrive/CP5403_A2/models/socratic_lora_v2


## 3. Load the retrieval index (no PDFs are processed here)

In [7]:
import faiss
from sentence_transformers import SentenceTransformer

index    = faiss.read_index(f"{INDEX_DIR}/cp5403.faiss")
chunks   = json.load(open(f"{INDEX_DIR}/chunks.json", encoding="utf-8"))
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")   # keep GPU memory for the LLM
print(f"Loaded {index.ntotal} vectors / {len(chunks)} chunks")
assert index.ntotal == len(chunks), "Index and chunks.json are out of sync – re-run 01"

# Quiz slides (multiple-choice options) make the tutor loop on "which option is correct?"
QUIZ  = re.compile(r"Quiz Question|\bQuestion\s+\d+\b.*?\bA[.)]\s", re.I | re.S)
QUIZ2 = re.compile(r"(?:^|\s)A\.\s.+?\sB\.\s.+?\sC\.\s", re.S)   # Week 9 style: options without "Quiz Question"

# Non-teaching pages: AI-use statements, thank-you and reference-only slides
NOISE = re.compile(r"AI[- ]use Acknowledg|Thank You\b.{0,40}Questions|^\s*References\b", re.I | re.S)

def is_quiz(text):
    head = text[:400]
    return bool(QUIZ.search(head) or QUIZ2.search(head) or NOISE.search(head))

from collections import Counter
quiz_chunks = [c for c in chunks if is_quiz(c["text"])]
print(f"Quiz chunks excluded from retrieval: {len(quiz_chunks)}")
print("By week:", Counter(c["week"] for c in quiz_chunks))
print("Practical chunks wrongly flagged as quiz:", sum(c.get("type") == "practical" for c in quiz_chunks))

def retrieve(query, k=TOP_K, exclude_quiz=True):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k * 5)          # over-fetch, then filter
    results = []
    for s, i in zip(scores[0], ids[0]):
        if i < 0:
            continue
        c = chunks[i]
        if exclude_quiz and is_quiz(c["text"]):
            continue
        results.append(dict(c, score=float(s)))
        if len(results) == k:
            break
    return results

# Quiz slides may print the correct option; remove such lines so the tutor cannot leak them
ANSWER_LINE = re.compile(r"(correct answer|answer)\s*[:：]\s*[A-D]\b.*", re.I)

def trim(text, n=CHUNK_WORDS):
    text = ANSWER_LINE.sub("", text)
    words = text.split()
    return " ".join(words[:n]) + (" ..." if len(words) > n else "")

def format_context(results):
    # Same layout as the training data: one '(Week N) text' line per chunk
    lines = []
    for r in results:
        label = "Course overview" if r["week"] == 0 else f"Week {r['week']}"
        lines.append(f"({label}) {trim(r['text'])}")
    return "\n".join(lines)

# check: Week 9 quiz pages should now be excluded
for q in ["How should I evaluate a multimodal model?",
          "What does the feed-forward layer do in a transformer?"]:
    print(f"\n❓ {q}")
    for r in retrieve(q):
        tag = "practical" if r.get("type") == "practical" else f"p{r['pages']}"
        print(f"   [Week {r['week']} {tag}] {r['score']:.2f}  {r['text'][:90]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 568 vectors / 568 chunks
Quiz chunks excluded from retrieval: 50
By week: Counter({4: 8, 1: 7, 2: 7, 3: 7, 9: 6, 5: 4, 6: 4, 8: 4, 7: 3})
Practical chunks wrongly flagged as quiz: 0

❓ How should I evaluate a multimodal model?
   [Week 7 p[37]] 0.39  Evaluation: Human and Automatic Checks Instruction-tuned models are often evaluated using 
   [Week 5 p[5, 6]] 0.37  Recap Raschka, Sebastian. Build A Large Language Model (From Scratch). Manning, 2024. ISBN

❓ What does the feed-forward layer do in a transformer?
   [Week 4 p[18]] 0.52  Why Feed-Forward Layers Are Needed Attention combines information from relevant tokens. Bu
   [Week 4 practical] 0.49  Practical 4 – Exercise 4.1 from the practical: Calculate and compare the number of paramet


## 4. Load Phi-3 Mini (4-bit) + the LoRA adapter
The adapter can be switched off with `model.disable_adapter()`, so **one loaded model** serves all four modes.

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)   # same tokenizer/template saved in 02
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
tokenizer.padding_side = "left"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto",
                                            torch_dtype=torch.float16, attn_implementation="eager")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
model.config.use_cache = True

# same system-role handling as 02
probe = [{"role": "system", "content": "SYSTEM_PROBE"}, {"role": "user", "content": "hi"}]
try:
    SYSTEM_SUPPORTED = "SYSTEM_PROBE" in tokenizer.apply_chat_template(probe, tokenize=False)
except Exception:
    SYSTEM_SUPPORTED = False

def prepare_messages(msgs):
    if SYSTEM_SUPPORTED or msgs[0]["role"] != "system":
        return msgs
    return [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}] + msgs[2:]

stop_ids = [tokenizer.eos_token_id]
end_id = tokenizer.convert_tokens_to_ids("<|end|>")
if isinstance(end_id, int) and end_id != tokenizer.unk_token_id:
    stop_ids.append(end_id)

print("System role supported:", SYSTEM_SUPPORTED)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

System role supported: True
GPU memory used: 2.4 GB


## 5. Prompt builder, generation and response clean-up
**Retrieval query for follow-ups:** short replies such as *"I don't know"* contain no topic, and wrong answers pull retrieval toward the wrong topic. The query therefore combines the **first student question of the conversation** with the current message.

**Logistics questions** ("which week ...") always include the hand-written course-overview chunk, so the answer is grounded instead of guessed.

**Generation:** greedy decoding **without** repetition penalty, matching the settings used in the 02 before/after check. (A repetition penalty of 1.1 made the small model avoid normal words and produce unsupported content.)

**Response clean-up** (the "Response Filter" in the architecture diagram): removes role labels, cuts off text where the model starts writing the student's next turn, and removes links (the slides contain none, so any link is invented).

In [9]:
from contextlib import nullcontext

def build_user_message(context, history, student_msg):
    hist = history[-2 * MAX_HISTORY_TURNS:]
    hist_text = "\n".join(f"{h['role']}: {h['text']}" for h in hist) or "(none)"
    return f"[Course context]\n{context}\n\n[Conversation so far]\n{hist_text}\n\n[Student]\n{student_msg}"

def retrieval_query(history, student_msg):
    # first student question of the conversation + current message,
    # so follow-ups and wrong answers do not pull retrieval off-topic
    students = [h["text"] for h in history if h["role"] == "Student"]
    return " ".join(students[:1] + [student_msg])

LOGISTICS = re.compile(r"\b(which|what)\s+week\b", re.I)
OVERVIEW  = next((c for c in chunks if c["source"] == "course_overview"), None)
if OVERVIEW is None:
    print("⚠️ course_overview chunk not found – re-run 01 Section 5 and 6")

URL = re.compile(r"<a\s[^>]*>.*?</a>|https?://\S+", re.I | re.S)

def clean_reply(text):
    text = re.sub(r"^\s*(Tutor|Assistant)\s*:\s*", "", text)
    text = re.split(r"\n\s*(?:Student|\[Student\])\s*:?", text)[0]   # model writing the student's turn
    text = URL.sub("", text)                                          # slides contain no links
    return re.sub(r"\s{2,}", " ", text).strip()

@torch.no_grad()
def generate(messages, use_adapter=True):
    prompt = tokenizer.apply_chat_template(prepare_messages(messages), tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True), ids["input_ids"].shape[1]

MODES = {"base": (False, False), "rag": (True, False), "ft": (False, True), "hybrid": (True, True)}

GUARD = re.compile(r"\b(?:write|draft|complete|finish)\b.{0,40}\b(?:my|our)\b.{0,40}"
                   r"\b(?:assessment|assignment|report|reflection|essay|discussion|section|practical|paragraph)", re.I)
GUARD_REPLY = ("I can't write assessment work for you to submit. I only have the CP5403 slides and practical guides, "
               "not your assessment instructions, so please check those on LearnJCU or ask your lecturer. "
               "I can help you understand any course concept you want to write about. Which one would you like to start with?")
def tutor_reply(history, student_msg, mode="hybrid"):
    use_rag, use_adapter = MODES[mode]
    if mode == "hybrid" and GUARD.search(student_msg):
        return {"reply": GUARD_REPLY, "mode": mode + "+guard", "in_scope": None, "sources": [],
                "retrieval_s": 0.0, "generation_s": 0.0, "prompt_tokens": 0,
                "words": len(GUARD_REPLY.split()), "questions": GUARD_REPLY.count("?")}
    t0 = time.time()
    if use_rag:
        results = retrieve(retrieval_query(history, student_msg))
        if OVERVIEW is not None and LOGISTICS.search(student_msg):
            results = [dict(OVERVIEW, score=1.0)] + results[:TOP_K - 1]
        context = format_context(results)
        in_scope = results[0]["score"] >= THRESHOLD
    else:
        results, context, in_scope = [], "(none)", None
    t_retrieval = time.time() - t0

    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_message(context, history, student_msg)}]
    t1 = time.time()
    raw, prompt_tokens = generate(messages, use_adapter)
    t_generation = time.time() - t1

    reply = clean_reply(raw)
    return {
        "reply": reply, "mode": mode, "in_scope": in_scope,
        "sources": [{"week": r["week"], "pages": r["pages"], "score": round(r["score"], 2)} for r in results],
        "retrieval_s": round(t_retrieval, 3), "generation_s": round(t_generation, 2),
        "prompt_tokens": prompt_tokens, "words": len(reply.split()), "questions": reply.count("?"),
    }

# quick checks
print(retrieval_query([{"role": "Student", "text": "What does the feed-forward layer do?"},
                       {"role": "Tutor", "text": "..."}], "I don't know."))
print("Logistics match:", bool(LOGISTICS.search("Which week covers pretraining?")))
print(clean_reply("Try this <a href='x'>link</a> https://youtube.com/abc now.\nStudent: ok"))

What does the feed-forward layer do? I don't know.
Logistics match: True
Try this now.


## 6. Tutor session (multi-turn state lives here, not in the model)

In [10]:
from datetime import datetime

class TutorSession:
    def __init__(self, mode="hybrid", verbose=True):
        self.mode, self.verbose = mode, verbose
        self.history, self.log = [], []

    def ask(self, student_msg):
        out = tutor_reply(self.history, student_msg, self.mode)
        self.history += [{"role": "Student", "text": student_msg}, {"role": "Tutor", "text": out["reply"]}]
        self.log.append({"student": student_msg, **out})
        if self.verbose:
            self.show(student_msg, out)
        return out["reply"]

    @staticmethod
    def show(student_msg, out):
        print(f"🧑‍🎓 STUDENT: {student_msg}")
        print(f"🧑‍🏫 TUTOR [{out['mode']}]: {out['reply']}")
        if out["sources"]:
            src = ", ".join(f"W{s['week']} p{s['pages']} ({s['score']})" for s in out["sources"])
            flag = "" if out["in_scope"] else "  ⚠️ weak match"
            print(f"   📚 sources: {src}{flag}")
        print(f"   ⏱ retrieval {out['retrieval_s']}s | generation {out['generation_s']}s | "
              f"{out['words']} words, {out['questions']} question(s)\n")

    def save(self, name="session"):
        path = f"{LOG_DIR}/{name}_{self.mode}_{datetime.now():%Y%m%d_%H%M}.json"
        json.dump(self.log, open(path, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
        print("💾 saved", path)

## 7. Scripted demo conversation
A realistic multi-turn session on a topic **not** in the validation set: the student starts with a question, gives a partly wrong answer, gets stuck, then reaches the idea.
📸 Screenshot this for the Week 9 presentation. (Student turns are fixed, so the demo is reproducible.)

In [11]:
demo = TutorSession(mode="hybrid")
for msg in [
    "What does the feed-forward layer do in a transformer?",
    "I think it decides which tokens to pay attention to.",
    "I don't know.",
    "So attention mixes information between tokens, and the feed-forward layer processes each token's vector on its own?",
]:
    demo.ask(msg)
demo.save("demo_feedforward")

🧑‍🎓 STUDENT: What does the feed-forward layer do in a transformer?
🧑‍🏫 TUTOR [hybrid]: Imagine a transformer that can attend to the words "the cat sat on the mat". Attention lets it focus on "sat" when predicting "mat". But what does the transformer still need to do with the words "the cat sat on the mat"?
   📚 sources: W4 p[18] (0.52), W4 p[21] (0.49)
   ⏱ retrieval 0.338s | generation 7.91s | 42 words, 1 question(s)

🧑‍🎓 STUDENT: I think it decides which tokens to pay attention to.
🧑‍🏫 TUTOR [hybrid]: That's attention, but the transformer still needs to turn "the cat sat on the mat" into a prediction. It has a final vector for "sat", but what does it need to turn that into a prediction?
   📚 sources: W4 p[18] (0.57), W4 p[39] (0.56)
   ⏱ retrieval 0.039s | generation 4.56s | 36 words, 1 question(s)

🧑‍🎓 STUDENT: I don't know.
🧑‍🏫 TUTOR [hybrid]: That's okay. Let's take it step by step. The transformer has a vector for "sat", and it needs to turn that into a prediction. Which layer in

## 8. Behaviour checks: logistics, out of scope, integrity
These should **not** get a Socratic question back in the same way:
- logistics → direct answer
- out of scope → say the slides don't cover it
- assessment request → refuse to write it, offer guidance

In [12]:
for q in ["Which week covers pretraining?",
          "What is the capital of France?",
          "Write my Assessment 2 discussion section for me."]:
    TutorSession(mode="hybrid").ask(q)

🧑‍🎓 STUDENT: Which week covers pretraining?
🧑‍🏫 TUTOR [hybrid]: Week 5 covers it: "Pretraining on unlabelled data". Try it on a weekly slideshow and see what happens when you train a model on the slides without any labels.
   📚 sources: W0 p[0] (1.0), W5 p[36] (0.58)
   ⏱ retrieval 0.023s | generation 4.08s | 29 words, 0 question(s)

🧑‍🎓 STUDENT: What is the capital of France?
🧑‍🏫 TUTOR [hybrid]: That's a simple geography question, not part of the CP5403 slides. A tutor specialising in the course would be able to help. Try searching "What is the capital of France?" online.
   📚 sources: W4 p[16] (0.12), W1 p[12] (0.11)  ⚠️ weak match
   ⏱ retrieval 0.017s | generation 5.27s | 31 words, 1 question(s)

🧑‍🎓 STUDENT: Write my Assessment 2 discussion section for me.
🧑‍🏫 TUTOR [hybrid+guard]: I can't write assessment work for you to submit. I only have the CP5403 slides and practical guides, not your assessment instructions, so please check those on LearnJCU or ask your lecturer. I can help 

## 9. Same question, four modes (preview of `04_evaluation`)
Shows what each component adds. 📸 Good slide material: one question, four answers.

In [13]:
question = "Why do we need positional embeddings?"
rows = []
for mode in ["base", "rag", "ft", "hybrid"]:
    out = tutor_reply([], question, mode)
    TutorSession.show(question, out)
    rows.append({k: out[k] for k in ["mode", "words", "questions", "generation_s"]})

import pandas as pd
pd.DataFrame(rows)

🧑‍🎓 STUDENT: Why do we need positional embeddings?
🧑‍🏫 TUTOR [base]: Positional embeddings are crucial in language models because they provide the model with information about the order of words in a sentence. Without this, the model wouldn't know which word comes first, second, and so on. Can you think of an example where the order of words changes the meaning of a sentence?
   ⏱ retrieval 0.0s | generation 3.99s | 53 words, 1 question(s)

🧑‍🎓 STUDENT: Why do we need positional embeddings?
🧑‍🏫 TUTOR [rag]: To understand why positional embeddings are necessary, let's first consider what an embedding is. Can you tell me what an embedding represents in the context of language models?
   📚 sources: W2 p[17, 18, 19] (0.55), W2 p[8] (0.45)
   ⏱ retrieval 0.026s | generation 2.3s | 28 words, 1 question(s)

🧑‍🎓 STUDENT: Why do we need positional embeddings?
🧑‍🏫 TUTOR [ft]: Imagine a word like "cat" and a word like "caterpillar". They sound the same at the start, but one is longer. If a model 

,mode,words,questions,generation_s
0,base,53,1,3.99
1,rag,28,1,2.30
2,ft,48,2,6.14
3,hybrid,43,2,5.55


## 10. Interactive chat
Type a question and press Enter. Commands: `reset` (new conversation), `save`, `quit`.

In [14]:
session = TutorSession(mode="hybrid", verbose=False)
print("CP5403 Socratic Tutor – type 'quit' to stop, 'reset' to start over, 'save' to save the transcript.\n")
while True:
    msg = input("🧑‍🎓 You: ").strip()
    if not msg:
        continue
    if msg.lower() == "quit":
        break
    if msg.lower() == "reset":
        session = TutorSession(mode="hybrid", verbose=False); print("🔄 New conversation\n"); continue
    if msg.lower() == "save":
        session.save("interactive"); continue
    session.ask(msg)
    TutorSession.show(msg, session.log[-1])

CP5403 Socratic Tutor – type 'quit' to stop, 'reset' to start over, 'save' to save the transcript.

🧑‍🎓 You: quit


## 11. Latency summary (supports the report claim that retrieval is negligible)

In [15]:
all_logs = demo.log + session.log
if all_logs:
    r = [l["retrieval_s"] for l in all_logs if l["sources"]]
    g = [l["generation_s"] for l in all_logs]
    print(f"Turns: {len(all_logs)}")
    print(f"Retrieval  mean {sum(r)/len(r):.3f}s")
    print(f"Generation mean {sum(g)/len(g):.2f}s")
    print(f"Mean prompt length: {sum(l['prompt_tokens'] for l in all_logs)/len(all_logs):.0f} tokens (limit 4096)")

Turns: 4
Retrieval  mean 0.110s
Generation mean 5.72s
Mean prompt length: 446 tokens (limit 4096)


## 12. What to check, and notes for the report
**Good signs**
- Demo turn 2 corrects the misconception without giving the answer; turn 4 confirms and extends
- Sources point to the right week (feed-forward → Week 4)
- Logistics question answered directly; France question flagged as weak match

**If something looks wrong**
| Symptom | Try |
|---|---|
| Tutor writes the student's next line | already cut by `clean_reply`; if it persists, lower `MAX_NEW_TOKENS` |
| Follow-up "I don't know" retrieves unrelated slides | check `sources`; the query already includes the previous student message |
| Prompt tokens close to 4096 | lower `MAX_HISTORY_TURNS` or `CHUNK_WORDS` |
| Replies repeat themselves | raise `repetition_penalty` to 1.2 |

**Report sentences**
- *Conversation state is managed at the application layer: the last four exchanges are inserted into the prompt in the same `[Conversation so far]` format used during fine-tuning.*
- *For follow-up turns, the retrieval query combines the current and previous student messages, so short replies such as "I don't know" still retrieve on-topic slides.*
- *Lines that reveal quiz answers are removed from retrieved chunks to avoid leaking answers through the context.*

**Next notebook:** `04_evaluation.ipynb` – the 15 held-out questions × four modes, with scoring.